
# 06 — GraphSAGE + LSTM Optuna Tuning
## Purpose
Tune the GraphSAGE + LSTM baseline with Optuna using the patch-store pipeline.
Report both global metrics and daytime-only metrics.

## Imports and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import time
import random
from functools import lru_cache
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, get_worker_info

import optuna

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

/srv/projects/Proyecto_e_ladino/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Paths & Optuna settings

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"
GROUND_DIR   = PROJECT_ROOT / "data" / "ground_aligned"
RUNS_ROOT    = PROJECT_ROOT / "runs" / "graphsage_lstm_optuna"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

SITE = "uniandes"
PATCH = 16
DAY_THRESHOLD = 20.0

DEBUG = False
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

# Optuna
N_TRIALS = 30
STUDY_NAME = f"graphsage_lstm_{SITE}_P{PATCH}"

DEVICE: cuda


## Load metadata and manifests

In [3]:
SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
PATCHES_ROOT = PROJECT_ROOT / "data" / "patches_v1" / SITE / f"P{PATCH}"
assert PATCHES_ROOT.exists(), f"Missing patches root: {PATCHES_ROOT}"

FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

Loaded dataset_meta.json:
{
  "site": "uniandes",
  "grid_size": 256,
  "site_center_pix": {
    "row": 160,
    "col": 49
  },
  "freq_min": 10,
  "horizon_steps": 36,
  "history_steps": 12,
  "mcmipf_root": "/srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF",
  "notes": "Manifest-only dataset. Satellite patches are extracted on-the-fly by the model."
}
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


In [4]:
def _hist_len(x):
    if isinstance(x, str):
        x = json.loads(x)
    elif isinstance(x, np.ndarray):
        x = x.tolist()
    return len(x)

L_manifest = int(train_man["history_ts"].map(_hist_len).mode().iloc[0])
L_meta = int(meta["history_steps"])
if L_manifest != L_meta:
    print(f"[WARN] meta L={L_meta} but manifest L={L_manifest}. Using manifest L.")
L = L_manifest

[WARN] meta L=12 but manifest L=24. Using manifest L.


## Reproducibility

In [5]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)

def seed_worker(worker_id: int) -> None:
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

SEED: 42


In [6]:
if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

DEBUG: False
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


## Persistence baseline

In [7]:
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

## Metrics

In [8]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray, day_threshold: float = 20.0) -> Dict[str, float]:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))

    day_mask = y_true >= day_threshold
    if day_mask.sum() > 0:
        rmse_day = float(np.sqrt(np.mean((y_true[day_mask] - y_pred[day_mask]) ** 2)))
        mae_day  = float(np.mean(np.abs(y_true[day_mask] - y_pred[day_mask])))
        n_day = int(day_mask.sum())
    else:
        rmse_day = float("nan")
        mae_day = float("nan")
        n_day = 0

    return {
        "n": int(len(y_true)),
        "rmse": rmse,
        "mae": mae,
        "n_day": n_day,
        "rmse_day": rmse_day,
        "mae_day": mae_day,
        "day_threshold": float(day_threshold),
    }

def skill_score(rmse_model: float, rmse_baseline: float) -> float:
    return float(1.0 - rmse_model / (rmse_baseline + 1e-12))

In [9]:
def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame, day_threshold: float = 20.0) -> Dict[str, float]:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()
    return metrics_from_arrays(y_true, y_hat, day_threshold=day_threshold)

baseline_train = eval_persistence(train_man, ground, day_threshold=DAY_THRESHOLD)
baseline_val   = eval_persistence(val_man, ground, day_threshold=DAY_THRESHOLD)
baseline_test  = eval_persistence(test_man, ground, day_threshold=DAY_THRESHOLD)

print("=== Persistence baseline ===")
print("train:", baseline_train)
print("val:  ", baseline_val)
print("test: ", baseline_test)

=== Persistence baseline ===
train: {'n': 54470, 'rmse': 419.8303181060779, 'mae': 281.04479528180656, 'n_day': 25215, 'rmse_day': 498.6695422994848, 'mae_day': 398.23299048185606, 'day_threshold': 20.0}
val:   {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656, 'n_day': 5467, 'rmse_day': 462.17602240706213, 'mae_day': 370.62950722516916, 'day_threshold': 20.0}
test:  {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714, 'n_day': 5465, 'rmse_day': 470.92567245567665, 'mae_day': 369.5877123513266, 'day_threshold': 20.0}


## Target normalization

In [10]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats (train):")
print("  mean:", Y_MEAN)
print("  std :", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())

Target stats (train):
  mean: 180.78839786820285
  std : 283.1064661153594
  percentiles: [0.0, 2.202, 628.0594000000003, 858.5459999999997, 1088.74274]


## Patch store

In [11]:
PATCH_HOUR_CACHE_MAXSIZE = 16

def patch_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh  = t.strftime("%H")
    year  = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_patch.npz"
    return PATCHES_ROOT / year / month / fname

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10  # 0..5

def patch_to_node_features(patch_c_pp: np.ndarray) -> np.ndarray:
    # (C, P, P) -> (P*P, C)
    C, P1, P2 = patch_c_pp.shape
    x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x

def load_patch_npz_nocache(path_str: str) -> np.ndarray:
    p = Path(path_str)
    with np.load(p) as d:
        arr = d["patch"]  # (6,16,P,P), float16 on disk
    return arr

@lru_cache(maxsize=PATCH_HOUR_CACHE_MAXSIZE)
def load_patch_npz_maincache(path_str: str) -> np.ndarray:
    return load_patch_npz_nocache(path_str)

def load_patch_npz(path_str: str) -> np.ndarray:
    if get_worker_info() is None:
        return load_patch_npz_maincache(path_str)
    return load_patch_npz_nocache(path_str)

## Datasets

In [12]:
class GraphSeqDataset(Dataset):
    """
    Returns:
      x_seq: torch.FloatTensor (L, N, C=16)
      y:     torch.FloatTensor scalar (normalized)
    """
    def __init__(self, manifest: pd.DataFrame):
        self.man = manifest.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = norm_y(float(row["y"]))

        history_ts = row["history_ts"]
        if isinstance(history_ts, str):
            history_ts = json.loads(history_ts)
        elif isinstance(history_ts, np.ndarray):
            history_ts = history_ts.tolist()

        xs = []
        for ts_str in history_ts:
            t = pd.to_datetime(ts_str, utc=True)
            p = patch_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            if not p.exists():
                raise FileNotFoundError(f"Missing patch file: {p}")

            tensor = load_patch_npz(str(p))   # (6,16,P,P)
            frame = tensor[slot]              # (16,P,P)
            frame = np.nan_to_num(frame, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

            x = patch_to_node_features(frame)  # (N,16)
            xs.append(x)

        x_seq = np.stack(xs, axis=0)  # (L,N,16)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = GraphSeqDataset(train_man)
val_ds   = GraphSeqDataset(val_man)
test_ds  = GraphSeqDataset(test_man)

print("datasets:", len(train_ds), len(val_ds), len(test_ds))

datasets: 54508 12407 12075


## Graph structure

In [13]:
def build_edge_index_8n(patch: int) -> torch.Tensor:
    edges = []

    def node_id(rr: int, cc: int) -> int:
        return rr * patch + cc

    for rr in range(patch):
        for cc in range(patch):
            u = node_id(rr, cc)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    r2, c2 = rr + dr, cc + dc
                    if 0 <= r2 < patch and 0 <= c2 < patch:
                        v = node_id(r2, c2)
                        edges.append((u, v))

    return torch.tensor(edges, dtype=torch.long).t().contiguous()

EDGE_INDEX = build_edge_index_8n(PATCH)
N_NODES = PATCH * PATCH

print("EDGE_INDEX:", EDGE_INDEX.shape)
print("N_NODES:", N_NODES)

EDGE_INDEX: torch.Size([2, 1860])
N_NODES: 256


## Dataloaders

In [14]:
def make_loader(ds: Dataset, batch_size: int, shuffle: bool, num_workers: int) -> DataLoader:
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(DEVICE == "cuda"),
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=(num_workers > 0),
    )
    if num_workers > 0:
        kwargs["prefetch_factor"] = 2
    return DataLoader(ds, **kwargs)

## Model

In [15]:
def batch_edge_index(edge_index: torch.Tensor, batch_size: int, num_nodes: int, device=None) -> torch.Tensor:
    if device is None:
        device = edge_index.device
    edge_index = edge_index.to(device)

    E = edge_index.size(1)
    offsets = (torch.arange(batch_size, device=device) * num_nodes).view(-1, 1)
    batched = edge_index.unsqueeze(0) + offsets.unsqueeze(-1)
    batched = batched.permute(1, 0, 2).reshape(2, batch_size * E)
    return batched.contiguous()

class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_nei  = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index[0], edge_index[1]
        N, F = x.shape

        nei_sum = torch.zeros((N, F), device=x.device, dtype=x.dtype)
        nei_sum.index_add_(0, dst, x[src])

        nei_cnt = torch.zeros((N, 1), device=x.device, dtype=x.dtype)
        nei_cnt.index_add_(0, dst, torch.ones((dst.numel(), 1), device=x.device, dtype=x.dtype))

        nei_mean = nei_sum / (nei_cnt + 1e-6)
        out = self.lin_self(x) + self.lin_nei(nei_mean)
        return torch.relu(out)

class GraphSAGE_LSTM(nn.Module):
    def __init__(self, in_dim: int, hidden_g: int, hidden_t: int, dropout_head: float = 0.0):
        super().__init__()
        self.g1 = GraphSAGELayer(in_dim, hidden_g)
        self.g2 = GraphSAGELayer(hidden_g, hidden_g)

        self.lstm = nn.LSTM(input_size=hidden_g, hidden_size=hidden_t, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Dropout(dropout_head),
            nn.Linear(hidden_t, 1),
        )

    def forward(self, x_seq: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        B, Ls, N, C = x_seq.shape
        device = x_seq.device

        be = batch_edge_index(edge_index, batch_size=B, num_nodes=N, device=device)

        embeds = []
        for t in range(Ls):
            x = x_seq[:, t].reshape(B * N, C)
            h = self.g1(x, be)
            h = self.g2(h, be)
            h = h.view(B, N, -1)
            g = h.mean(dim=1)
            embeds.append(g)

        z = torch.stack(embeds, dim=1)
        out, _ = self.lstm(z)
        last = out[:, -1]
        yhat = self.head(last).squeeze(-1)
        return yhat

## Training

In [16]:
@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader, day_threshold: float = 20.0) -> Dict[str, float]:
    model.eval()
    ys, yhats = [], []

    edge_index_dev = EDGE_INDEX.to(DEVICE)

    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        yhat = model(x_seq, edge_index_dev)

        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    y_phys = denorm_y(y)
    yhat_phys = denorm_y(yhat)

    return metrics_from_arrays(y_phys, yhat_phys, day_threshold=day_threshold)

In [17]:
def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float,
    weight_decay: float,
    grad_clip_norm: float,
    use_amp: bool,
    epochs: int,
    patience: int,
    min_delta: float,
    day_threshold: float,
) -> Dict[str, Any]:

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    train_log = []
    best_val_rmse_day = float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0

    t_train0 = time.time()

    edge_index_dev = EDGE_INDEX.to(DEVICE)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_losses = []

        for x_seq, y in train_loader:
            x_seq = x_seq.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                yhat = model(x_seq, edge_index_dev)
                loss = loss_fn(yhat, y)

            scaler.scale(loss).backward()

            if grad_clip_norm is not None and grad_clip_norm > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(opt)
            scaler.update()

            tr_losses.append(loss.item())

        val_metrics = eval_model(model, val_loader, day_threshold=day_threshold)
        val_rmse = float(val_metrics["rmse"])
        val_rmse_day = float(val_metrics["rmse_day"])
        val_mae = float(val_metrics["mae"])
        val_mae_day = float(val_metrics["mae_day"])

        scheduler.step(val_rmse_day)

        epoch_out = {
            "epoch": epoch,
            "train_mse_norm": float(np.mean(tr_losses)),
            "val_rmse_phys": val_rmse,
            "val_mae_phys": val_mae,
            "val_rmse_day_phys": val_rmse_day,
            "val_mae_day_phys": val_mae_day,
            "lr": float(opt.param_groups[0]["lr"]),
            "epoch_seconds": float(time.time() - t0),
        }
        train_log.append(epoch_out)

        improved = (best_val_rmse_day - val_rmse_day) > min_delta
        if improved:
            best_val_rmse_day = val_rmse_day
            best_epoch = epoch
            bad_epochs = 0
            best_state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "best_epoch": best_epoch,
                "best_val_rmse_day": best_val_rmse_day,
            }
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    total_train_seconds = float(time.time() - t_train0)

    assert best_state is not None, "No best state captured during training."

    model.load_state_dict(best_state["model_state"])

    final_val = eval_model(model, val_loader, day_threshold=day_threshold)

    return {
        "model": model,
        "best_epoch": int(best_state["best_epoch"]),
        "best_val_rmse_day": float(best_state["best_val_rmse_day"]),
        "final_val": final_val,
        "train_log": train_log,
        "train_seconds_total": total_train_seconds,
    }

## Optuna

### Objective

In [18]:
USE_AMP = (DEVICE == "cuda")
GRAD_CLIP_NORM = 1.0
EPOCHS = 20
PATIENCE = 6
MIN_DELTA = 0.0

In [19]:
def objective(trial: optuna.Trial) -> float:
    hidden_g = trial.suggest_categorical("hidden_g", [32, 64, 96, 128])
    hidden_t = trial.suggest_categorical("hidden_t", [32, 64, 96, 128])
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_head = trial.suggest_categorical("dropout_head", [0.0, 0.1, 0.2, 0.3])

    train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = make_loader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = GraphSAGE_LSTM(
        in_dim=16,
        hidden_g=hidden_g,
        hidden_t=hidden_t,
        dropout_head=dropout_head,
    ).to(DEVICE)

    out = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=lr,
        weight_decay=weight_decay,
        grad_clip_norm=GRAD_CLIP_NORM,
        use_amp=USE_AMP,
        epochs=EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        day_threshold=DAY_THRESHOLD,
    )

    val_metrics = out["final_val"]
    n_params = sum(p.numel() for p in model.parameters())

    trial.set_user_attr("best_epoch", out["best_epoch"])
    trial.set_user_attr("val_rmse", float(val_metrics["rmse"]))
    trial.set_user_attr("val_mae", float(val_metrics["mae"]))
    trial.set_user_attr("val_rmse_day", float(val_metrics["rmse_day"]))
    trial.set_user_attr("val_mae_day", float(val_metrics["mae_day"]))
    trial.set_user_attr("train_seconds_total", float(out["train_seconds_total"]))
    trial.set_user_attr("n_params", int(n_params))

    return float(val_metrics["rmse_day"])

## Run

### Optuna

In [20]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

[I 2026-03-10 09:47:47,822] A new study created in memory with name: graphsage_lstm_uniandes_P16


  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_707136/4166800974.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


/tmp/ipykernel_707136/4166800974.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


  0%|          | 0/30 [21:14<?, ?it/s]

...

Best trial: 17. Best value: 281.019:  97%|█████████▋| 29/30 [17:00:28<35:00, 2100.04s/it]

Best trial: 17. Best value: 281.019:  97%|█████████▋| 29/30 [17:00:28<35:00, 2100.04s/it]

Best trial: 17. Best value: 281.019: 100%|██████████| 30/30 [17:00:28<00:00, 2271.34s/it]

Best trial: 17. Best value: 281.019: 100%|██████████| 30/30 [17:00:28<00:00, 2040.95s/it]

[I 2026-03-11 02:48:16,350] Trial 29 finished with value: 287.38388026098175 and parameters: {'hidden_g': 64, 'hidden_t': 128, 'lr': 0.00020972308408885693, 'batch_size': 16, 'weight_decay': 0.0005051820468833849, 'dropout_head': 0.2}. Best is trial 17 with value: 281.01911642614334.


In [21]:
print("Best trial:")
print("  value (val_rmse_day):", study.best_trial.value)
print("  params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

Best trial:
  value (val_rmse_day): 281.01911642614334
  params:
    hidden_g: 32
    hidden_t: 96
    lr: 0.00010905817062217421
    batch_size: 8
    weight_decay: 0.00012148461943149346
    dropout_head: 0.1


## Summary

### Trials

In [22]:
rows = []
for t in study.trials:
    row = {
        "number": t.number,
        "state": str(t.state),
        "objective_val_rmse_day": t.value,
    }
    row.update(t.params)
    row.update(t.user_attrs)
    rows.append(row)

trials_df = pd.DataFrame(rows).sort_values("objective_val_rmse_day", ascending=True)
trials_df.head(10)

,number,state,objective_val_rmse_day,hidden_g,hidden_t,lr,batch_size,weight_decay,dropout_head,best_epoch,val_rmse,val_mae,val_rmse_day,val_mae_day,train_seconds_total,n_params
17,17,TrialState.COMPLETE,281.019116,32,96,0.000109,8,0.000121,0.1,10,228.451003,168.170113,281.019116,218.046244,2812.499847,62529
20,20,TrialState.COMPLETE,282.074397,32,64,0.000214,8,0.000025,0.1,11,261.705353,207.624781,282.074397,233.412038,2992.649523,32513
10,10,TrialState.COMPLETE,282.649393,32,96,0.000103,8,0.000068,0.2,2,243.221710,197.040330,282.649393,221.702101,1409.421925,62529
23,23,TrialState.COMPLETE,283.579917,32,64,0.000104,8,0.000024,0.1,3,244.241561,193.241373,283.579917,225.590346,1582.229165,32513
11,11,TrialState.COMPLETE,283.786117,32,96,0.000112,8,0.000053,0.2,9,251.994718,196.054144,283.786117,233.283734,2637.633029,62529
21,21,TrialState.COMPLETE,286.522472,32,64,0.000175,8,0.000014,0.1,4,238.229804,180.277622,286.522472,229.640407,1755.446038,32513
2,2,TrialState.COMPLETE,287.040208,32,32,0.000112,8,0.000009,0.3,6,228.999674,165.546673,287.040208,224.127449,2112.640708,12737
22,22,TrialState.COMPLETE,287.334352,32,64,0.000219,8,0.000032,0.1,15,220.733414,153.851253,287.334352,218.575559,3513.648179,32513
29,29,TrialState.COMPLETE,287.383880,64,128,0.000210,16,0.000505,0.2,9,235.092918,171.981024,287.383880,224.409213,2605.346528,126465
6,6,TrialState.COMPLETE,287.691649,96,64,0.000268,16,0.000079,0.3,11,240.328047,179.360028,287.691649,226.755873,2969.016459,67585


## Retrain

In [23]:
best_params = study.best_trial.params
best_params

{'hidden_g': 32,
 'hidden_t': 96,
 'lr': 0.00010905817062217421,
 'batch_size': 8,
 'weight_decay': 0.00012148461943149346,
 'dropout_head': 0.1}

In [24]:
FINAL_EPOCHS = 30
FINAL_PATIENCE = 8

best_batch_size = best_params["batch_size"]

train_loader = make_loader(train_ds, batch_size=best_batch_size, shuffle=True, num_workers=4)
val_loader   = make_loader(val_ds, batch_size=best_batch_size, shuffle=False, num_workers=0)
test_loader  = make_loader(test_ds, batch_size=best_batch_size, shuffle=False, num_workers=0)

best_model = GraphSAGE_LSTM(
    in_dim=16,
    hidden_g=best_params["hidden_g"],
    hidden_t=best_params["hidden_t"],
    dropout_head=best_params["dropout_head"],
).to(DEVICE)

out = train_one_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    grad_clip_norm=GRAD_CLIP_NORM,
    use_amp=USE_AMP,
    epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    min_delta=MIN_DELTA,
    day_threshold=DAY_THRESHOLD,
)

final_val = out["final_val"]
final_test = eval_model(best_model, test_loader, day_threshold=DAY_THRESHOLD)

final_val["skill_vs_persistence"] = skill_score(final_val["rmse"], baseline_val["rmse"])
final_val["skill_day_vs_persistence"] = skill_score(final_val["rmse_day"], baseline_val["rmse_day"])

final_test["skill_vs_persistence"] = skill_score(final_test["rmse"], baseline_test["rmse"])
final_test["skill_day_vs_persistence"] = skill_score(final_test["rmse_day"], baseline_test["rmse_day"])

print("Best tuned GraphSAGE val:", final_val)
print("Best tuned GraphSAGE test:", final_test)

/tmp/ipykernel_707136/4166800974.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


/tmp/ipykernel_707136/4166800974.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Best tuned GraphSAGE val: {'n': 12407, 'rmse': 235.59620167761219, 'mae': 177.11146920761362, 'n_day': 5468, 'rmse_day': 286.6627711040193, 'mae_day': 224.99087439860057, 'day_threshold': 20.0, 'skill_vs_persistence': 0.37727184887750054, 'skill_day_vs_persistence': 0.37975412568776645}
Best tuned GraphSAGE test: {'n': 12075, 'rmse': 245.09533880054656, 'mae': 185.85355762300037, 'n_day': 5465, 'rmse_day': 288.17620898535955, 'mae_day': 235.40750579415172, 'day_threshold': 20.0, 'skill_vs_persistence': 0.3707725012735664, 'skill_day_vs_persistence': 0.38806434679459534}


## Save outputs

In [25]:
run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "run_name": run_name,
    "study_name": STUDY_NAME,
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "day_threshold": DAY_THRESHOLD,
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "patches_root": str(PATCHES_ROOT),
        "ground_path": str(ground_path),
        "run_dir": str(RUN_DIR),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "nodes": int(PATCH * PATCH),
        "channels": 16,
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "optuna": {
        "n_trials": int(N_TRIALS),
        "best_value_val_rmse_day": float(study.best_trial.value),
        "best_params": study.best_trial.params,
    },
    "best_model": {
        "arch": "GraphSAGE(2) + mean pool + LSTM + MLP head",
        "best_epoch": int(out["best_epoch"]),
        "train_seconds_total": float(out["train_seconds_total"]),
        "final_val": final_val,
        "final_test": final_test,
        "n_params": int(sum(p.numel() for p in best_model.parameters())),
    },
}

with open(RUN_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

trials_df.to_csv(RUN_DIR / "trials.csv", index=False)

print("Saved summary to:", RUN_DIR / "summary.json")
print("Saved trials to :", RUN_DIR / "trials.csv")

Saved summary to: /srv/projects/Proyecto_e_ladino/runs/graphsage_lstm_optuna/uniandes_H36_L24_P16_seed42_20260311_083135/summary.json
Saved trials to : /srv/projects/Proyecto_e_ladino/runs/graphsage_lstm_optuna/uniandes_H36_L24_P16_seed42_20260311_083135/trials.csv
